# Module 0: Vectors, length and the inner product

How would you tell a computer where your Roblox character is standing, how strong a Knight is, or what color a sunset is? You would give it a short list of numbers, and that list has a name: a **vector**. In this notebook you will draw vectors, add them, measure how long they are, and find out how much two of them point the same way.

You will learn the way scientists do: by guessing first. Before you run a cell, say out loud what you think it will print or draw. Some of your guesses will be wrong, and those are the best ones, because a surprise is the moment you learn something new.

To run a cell, click on it and press **Shift + Enter**. Start with the two setup cells below, once each, in order.

**Keep your work.** A notebook opened in Google Colab from the course page is only borrowed, and Colab forgets your changes when you close it. To keep them, choose **File**, then **Save a copy in Drive**, and work in your copy. Then you can come back later and see everything you did.

<small>Copyright © 2026-present CAehi.us. This whole notebook, text, code and pictures, is licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).</small>

<p><a href="https://colab.research.google.com/github/frcusaca/admath/blob/jia/notebooks/01_rover_moves_linear_combinations.ipynb" target="_blank" rel="noopener" title="Next: Notebook 1: The Rover&#x27;s Moves (linear combinations) (opens in a new tab)">Notebook 1 <b style="color:#a8327d">&gt;&gt;</b></a></p>

**Setup.** The next cell gets the drawing library, `vecdraw`. The library is short, and you are allowed to read it and change it. To read all of it, run `import inspect` and then `print(inspect.getsource(vecdraw))`.

In [ ]:
# Setup: this cell gets the vecdraw library. You do not need to read it.
REPO_URL = "https://github.com/frcusaca/admath"   # set in course.toml
REPO_REF = "jia"

import importlib.util, sys
from pathlib import Path

if "google.colab" in sys.modules:
    if importlib.util.find_spec("vecdraw") is None:
        if "CHANGE-ME" in REPO_URL:
            raise RuntimeError("The course repository URL is not set yet. Ask your teacher.")
        %pip install --quiet "vecdraw @ git+{REPO_URL}@{REPO_REF}#subdirectory=python"
else:
    # On a computer with the course folder: use the library in python/src.
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "python" / "src" / "vecdraw").is_dir():
            sys.path.insert(0, str(folder / "python" / "src"))
            break
    if importlib.util.find_spec("vecdraw") is None:
        raise RuntimeError("Cannot find vecdraw. Run `make setup` in the course folder, or `pip install -e python`.")

%reload_ext vecdraw._predict

In [ ]:
import numpy as np
import importlib, vecdraw
importlib.reload(vecdraw)      # pick up any changes you make to the library
from vecdraw import *

---

## Part 0.0: What is a vector?

Computers only understand numbers. So how would you tell a computer about a color? A place in Roblox? A pizza order? A game character?

One number is not enough. "The Knight is a 9" tells you almost nothing. But a **list** of numbers can say a lot.

$$\text{Knight} = (9, 3, 8, 1, 5)$$

The five slots mean (strength, stealth, armor, magic, luck), in that order. Now you know that this Knight hits hard, wears heavy armor and is hopeless at magic.

That list is a **vector**.

> A **vector** is a list of numbers in which each slot always means the same thing.

Once you start looking, vectors are everywhere.

| What it describes | What the slots mean | Example |
| --- | --- | --- |
| a spot on a map | (steps east, steps north) | $(3, 4)$ |
| a place in Roblox | $(x, y, z)$ in studs, and $y$ is up | $(10, 0, 5)$ |
| a color on a screen | (red, green, blue), each from 0 to 255 | $(255, 128, 0)$ is orange |
| a game character | (strength, stealth, armor, magic, luck) | $(9, 3, 8, 1, 5)$ |
| a pizza order | (pepperoni, mushroom, olive) | $(2, 1, 0)$ |
| a snack bar | (calories, protein, fat, carbs, sugar) | $(200, 10, 8, 25, 12)$ |
| a day of weather | (high in °F, low in °F, rain in mm) | $(68, 51, 4)$ |
| a calculator digit | seven bars: 1 is lit, 0 is dark | $(1, 1, 1, 0, 0, 0, 0)$ is the digit 7 |
| a doctor's chart | (weight in kg, height in m, BMI) | $(40, 1.5, 17.8)$ |

Your ML class calls these **feature vectors**. Every model you train eats vectors.

### The order is a promise

$(3, 4)$ and $(4, 3)$ use the same two numbers, but they are different spots on the map. $(255, 0, 0)$ is pure red, and $(0, 0, 255)$ is pure blue.

So a vector is really two things: the numbers, **and** the agreement about what each slot means. The agreement is not stored inside the vector. You have to know it. Good programmers write it down in a comment.

We write vectors sideways, like $(3, 4)$. Some books write them top to bottom, like $\genfrac{(}{)}{0pt}{0}{3}{4}$. It is the same vector. Sideways just saves paper.

### How many slots?

The number of slots is called the **dimension** of the vector. A map spot is 2-dimensional. A color is 3-dimensional. The Knight is 5-dimensional, and a calculator digit is 7-dimensional.

Nobody can draw a picture in 5 dimensions. It does not matter. Almost everything in this course works for any number of slots. Vectors with 2 or 3 slots are special for one reason only: we can draw them.

### What are vectors good for?

You can do arithmetic on a whole list at once.

- Two <span title="A pizza order is (pepperoni, mushroom, olive). Check the table above." style="text-decoration: underline dotted; cursor: help">pizza orders</span>, $(2, 1, 0)$ on Monday and $(1, 0, 3)$ on Tuesday, add up slot by slot to $(3, 1, 3)$. This is called **element-wise** addition.
- A pancake recipe of (2 cups of flour, 3 eggs, 1 cup of milk) doubles to $(4, 6, 2)$.

**A new word: element-wise.** Each number in a vector is called an **element**. *Element-wise* is two words joined by a hyphen (-), and it works as an adjective: it describes something, the way *left-handed* or *well-known* do. It means "one element at a time, each with the element in the same slot". Element-wise addition adds the first slots together, then the second slots, and so on. Nothing ever moves to a different slot.

Not every list of numbers deserves the name. A phone number is a list of digits, but adding two phone numbers gives nonsense. A list earns the name **vector** when adding two of them slot by slot, or doubling one, means something.

### Two things to keep an eye on

**Slots can have different units.** The snack bar mixes calories and grams. That is allowed. It will need some care later, when we ask how long a vector is.

**Some slots tell you nothing new.** Look at the doctor's chart. BMI is worked out from the weight and the height with a formula. If you know the first two slots, you can fill in the third one yourself. So is that slot really adding anything? Hold that thought. It comes back in a big way in Module 3.

### First, Python is a calculator

Predict each answer, then run the cell.

In [ ]:
print(3 + 4)
print(10 / 4)
print(2 ** 5)      # ** means "to the power of"

Some cells keep their answer a secret until you have made your guess. A line that starts with `%guess_first` only asks for your guess the first time you run it. Run the cell a second time, and it shows the answer.

What is `2 ** 10`? Guess, run the cell, then run it again.


In [ ]:
%guess_first 2 ** 10


A **variable** is a named box that holds a value. Anything after a `#` is a note for humans. Python ignores it.

In [ ]:
speed = 5              # meters per second
time = 3               # seconds
distance = speed * time
print(distance)        # meters

**Your turn.** The cell below starts as a copy of the one above, and it is yours to change. Give `speed` and `time` new values and run it. Then make a new variable called `laps`, and print it.


In [ ]:
speed = 5              # meters per second: change me
time = 3               # seconds: change me
distance = speed * time
print(distance)        # meters

laps = 0               # change me
print(laps)


### Your first vectors

In NumPy we make a vector with `np.array`. The comment keeps the promise about what each slot means.

Predict what each line prints. Careful: Python starts counting at 0!

In [ ]:
knight = np.array([9, 3, 8, 1, 5])     # (strength, stealth, armor, magic, luck)
print(knight)
print(knight[0])        # the first slot
print(knight[2])        # which stat is this?
print(len(knight))      # how many slots: the dimension

A color is a vector with three slots. In the next cell each slot goes from 0 to 1 instead of 0 to 255. What color do you expect?


In [ ]:
import matplotlib.pyplot as plt

color = np.array([1.0, 0.5, 0.0])      # red, green, blue
plt.figure(figsize=(2, 2))
plt.imshow([[color]])                  # one big pixel
plt.axis("off")
plt.show()

**Your turn.** Here is the same color again. Swap its first and last numbers and run the cell. The numbers are the same, only the order changed. Is it the same color?


In [ ]:
color = np.array([1.0, 0.5, 0.0])      # red, green, blue: swap the first and last numbers
plt.figure(figsize=(2, 2))
plt.imshow([[color]])
plt.axis("off")
plt.show()


A calculator digit is a vector with seven slots, one for each bar: top, upper right, lower right, bottom, lower left, upper left, middle.

Before you run it: which digit is this?

In [ ]:
digit = np.array([1, 1, 1, 0, 0, 0, 0])
draw_digit(digit)
show()

**Your turn.** The cell below draws the 7 again. Change the seven numbers to make a 4. Then make your favorite digit.


In [ ]:
digit = np.array([1, 1, 1, 0, 0, 0, 0])     # a 7: change the numbers to make a 4
draw_digit(digit)
show()


Now some arithmetic on whole lists. Predict both answers.


In [ ]:
monday = np.array([2, 1, 0])      # pepperoni, mushroom, olive
tuesday = np.array([1, 0, 3])
print(monday + tuesday)

pancakes = np.array([2, 3, 1])    # cups of flour, eggs, cups of milk
print(2 * pancakes)

### How Mastery Coins work

After each part there are ten or more Mastery Coins. Do them on paper, unless a coin says **(code)**.

- Coins 1 to 5 are **bronze**, worth 1 coin each.
- Coins 6 to 8 are **silver**, worth 2 coins each.
- Coins 9 and up are **gold**, worth 3 coins each.

You earn a coin when your answer is right **and** you can explain it to your partner.

### Mastery Coins 0.0

**Bronze (1 coin each)**

#### M 0.0.1

A color is (red, green, blue), each from 0 to 255. Orange is $(255, 128, 0)$. How much green is in it? How much blue?

#### M 0.0.2

A Roblox character is at $(10, 4, -5)$. The slots are $(x, y, z)$, and $y$ is up. How many studs above the ground is it?

#### M 0.0.3

The Knight is $(9, 3, 8, 1, 5)$. The slots are (strength, stealth, armor, magic, luck). What is its armor? Which stat is its weakest?

#### M 0.0.4

Give the dimension of each: a spot on a map, a color, the Knight, a calculator digit.

#### M 0.0.5

**(code)** Make a NumPy vector with at least 4 slots about something you like: a Pokémon, a car, your Roblox avatar. Write what each slot means in a comment. Print the whole vector, then print only its third slot.

**Silver (2 coins each)**

#### M 0.0.6

Are $(3, 4)$ and $(4, 3)$ the same spot on a map? What colors are $(255, 0, 0)$ and $(0, 0, 255)$? What do these two examples tell you about vectors?

#### M 0.0.7

A calculator digit has seven bars, in this order: top, upper right, lower right, bottom, lower left, upper left, middle. A lit bar is 1 and a dark bar is 0. The digit 7 is $(1, 1, 1, 0, 0, 0, 0)$. Write the vectors for the digits 1 and 8. Which digit is $(1, 1, 1, 1, 1, 1, 0)$?

#### M 0.0.8

Pizza orders are (pepperoni, mushroom, olive). Monday's order is $(2, 1, 0)$ and Tuesday's is $(1, 0, 3)$. Write the total for both days as one vector. Wednesday's order is double Monday's. Write it.

**Gold (3 coins each)**

#### M 0.0.9

For each pair, decide whether adding the two lists slot by slot means something: two shopping carts written as (apples, bananas, carrots); two phone numbers; two beams of colored light written as (red, green, blue); two locker combinations. What do the lists that work have in common?

#### M 0.0.10

A doctor's chart is (weight, height, BMI). BMI is worked out from weight and height with a formula. Clinic A writes down all three slots. Clinic B writes down only (weight, height). Which clinic knows more about its patients? Now invent another vector that has a slot which tells you nothing new.

In [ ]:
# Scratch space for the (code) coin in Part 0.0

---

## Part 0.1: Vectors as moves

From here on, most of the problems happen in two places.

- **The car map (2D).** A flat map. A spot is $(x, y)$: $x$ steps east and $y$ steps north. A car can only drive the way it is pointing.
- **The Roblox world (3D).** A spot is $(x, y, z)$, measured in studs. Just like in Roblox Studio, $y$ is **up**. Your character can walk one way while looking another way.

When the slots of a vector are steps in space, you can draw it. It becomes an **arrow**, and the arrow is a move.

A **point** is a place. A **vector** is a move. Letters that name points and vectors are printed in **bold**, like $\mathbf{P}$. Part 0.2 tells you why.

The move from a point $\mathbf{P}$ to a point $\mathbf{Q}$ is

$$\mathbf{Q} - \mathbf{P}$$

worked out one slot at a time. The move from $\mathbf{P} = (1, 2)$ to $\mathbf{Q} = (4, 6)$ is $(4 - 1, 6 - 2) = (3, 4)$.

### One list of numbers, two ways to read it

The numbers $(1, 2, 3)$ can be read in two ways.

- As a **point**: the spot you reach when you start at home, $(0, 0, 0)$, and walk one slot at a time: 1 step along $x$, then 2 steps along $y$, then 3 steps along $z$.
- As a **vector**: the move itself. Make the same three walks from wherever you are standing.

The two readings fit together. The point $(1, 2, 3)$ is where the vector $(1, 2, 3)$ takes you **when you start at home**. So when nobody says where an arrow starts, it starts at home, $(0, 0, 0)$, and its tip is the point $(1, 2, 3)$.

**Which way is it headed?** A move points somewhere. A car that drives along the move $(3, 4)$ is **heading in the direction of** $(3, 4)$: east and north, a little more north than east. The moves $(6, 8)$ and $(30, 40)$ are longer, but a car driving along them is heading in the same direction.

The word *heading* is used in more than one way, so here is how this course uses it. A ship's captain who orders a heading gives an angle, the number of degrees to turn from north on the compass. People also often call a car's velocity its heading. In these notebooks, something is *heading in the direction of* a vector, and that vector can be any length. In Part 0.3, once vectors have a length, **the heading** will mean one special vector for that direction: the one of length 1.

That is why NumPy uses `np.array` for both. Roblox does the same: a part's `Position` and its velocity are both a `Vector3`.

Sometimes the start has to be given. `draw_vector(v, start=P)` needs a start to draw an arrow that does not begin at home. A Roblox raycast needs two things: the point where the ray starts, and the direction it goes.

| What you do | What you get | Example |
| --- | --- | --- |
| point $-$ point | a vector | station $-$ car is the move from the car to the station |
| point $+$ vector | a point | car $+$ move is where the car ends up |
| vector $+$ vector | a vector | one move, then another |

NumPy will happily add two points too, but the answer rarely means anything. What would "my house plus the school" be?

Make the move $(3, 4)$ in NumPy and look at its two slots.

In [ ]:
a = np.array([3, 4])     # 3 steps east, 4 steps north
print(a)
print(a[0])              # the first number. Python starts counting at 0!
print(a[1])              # the second number

Now draw it. `new_grid(6)` gets a sheet of graph paper that goes from $-6$ to $6$.

In [ ]:
new_grid(6)
draw_vector(a, label="a")
show()

A car is parked at $\mathbf{P}$. The gas station is at $\mathbf{Q}$. Predict the move from the car to the station, then run.

In [ ]:
P = np.array([1, 2])       # the car
Q = np.array([4, 6])       # the gas station
move = Q - P
print(move)

new_grid(8)
draw_point(P, "car")
draw_point(Q, "station")
draw_vector(move, start=P, label="move", label_at="middle")
show()

The same move can start anywhere. It is still the same vector.

In [ ]:
new_grid(8)
draw_vector(move, label="from home")
draw_vector(move, start=(-6, -5), label="same move")
show()

In the Roblox world a vector has three numbers. Predict the move from you to the NPC.

In [ ]:
me = np.array([10, 0, 5])       # x, y, z in studs.  y is up.
npc = np.array([22, 0, 14])
print(npc - me)

**Your turn.** Make a vector `b` that goes 2 steps west and 5 steps north. Draw `a` and `b` on the same grid.

In [ ]:
b = np.array([0, 0])     # change me
new_grid(6)
draw_vector(a, label="a")
draw_vector(b, label="b")
show()

### Mastery Coins 0.1

**Bronze (1 coin each)**

#### M 0.1.1

Draw the vector $(4, 3)$ starting at home, $(0, 0)$. Where is its tip?

#### M 0.1.2

Draw the vector $(4, 3)$ starting at $(1, 1)$. Where is its tip?

#### M 0.1.3

A car is at $\mathbf{P} = (2, 1)$. The gas station is at $\mathbf{Q} = (5, 5)$. Write the move from the car to the station.

#### M 0.1.4

Now write the move from the station back to the car.

#### M 0.1.5

In Roblox you stand at $(10, 0, 5)$. An NPC stands at $(22, 0, 14)$. Write the move from you to the NPC.

**Silver (2 coins each)**

#### M 0.1.6

A 3D printer nozzle starts at $(0, 0, 0)$. It moves $(120, 40, 0)$, then $(-30, 25, 0.2)$. Where is it now? What single move sends it home?

#### M 0.1.7

You are at $(100, 64, -200)$. A diamond is at $(112, 12, -188)$. Write the move. How many blocks down is it?

#### M 0.1.8

**(code)** Make `v = np.array([7, -2, 5])` and print `v[0]` and `v[2]`. Then make the points $(3, 7)$ and $(3, 2)$, and print the move from the first to the second. Which way does that move point?

**Gold (3 coins each)**

#### M 0.1.9

The move from $\mathbf{P}$ to $\mathbf{Q}$ is $(6, -2)$, and $\mathbf{Q} = (10, 4)$. Where is $\mathbf{P}$?

#### M 0.1.10

$\mathbf{A} = (1, 1)$, $\mathbf{B} = (4, 5)$ and $\mathbf{C} = (7, 9)$. Write the move from $\mathbf{A}$ to $\mathbf{B}$ and the move from $\mathbf{B}$ to $\mathbf{C}$. What does that tell you about the three points?

In [ ]:
# Scratch space for the (code) coins in Part 0.1

---

## Part 0.2: Adding, subtracting and scaling

**Adding** two vectors means: make one move, then the other. In numbers, it is element-wise addition: add one slot at a time.

$$(3, 4) + (1, -2) = (4, 2)$$

**Subtracting** is element-wise too: subtract one slot at a time.

$$(3, 4) - (1, -2) = (2, 6)$$

**Counting copies.** "5 apples" means you can count 5 apples.

In math, $5x$ means 5 copies of $x$, all added together:

$$5x = x + x + x + x + x$$

We read $5x$ out loud as "five x". It is just like "five apples", only without the *s* sound at the end. Nothing goes between the $5$ and the $x$: writing them side by side is enough.

Negative numbers work too. $-5$ is a number, so $-5x$, read "negative five x", means $-5$ copies of $x$. Negative copies are copies of the opposite, $-x$:

$$-5x = (-x) + (-x) + (-x) + (-x) + (-x)$$

**A number with a nickname.** Now say there is a number $c$. It is some number. It could be $5$. It could be $-5$. When we want to talk about that number without saying which one it is, we call it by its nickname, $c$.

So instead of "5 apples" we can say "$c$ apples". Instead of $5x$ we can write $cx$, read "c x": $c$ copies of $x$.

A second number gets a second nickname, $d$, and there can be a second thing, $y$. Then

$$cx + dy$$

means $c$ copies of $x$ plus $d$ copies of $y$. "3 apples and 2 pears" is $cx + dy$ with $c = 3$, $x$ = an apple, $d = 2$ and $y$ = a pear.

**Vectors are bold.** From now on a letter that names a vector is printed **bold**: $\mathbf{a}$, $\mathbf{x}$, $\mathbf{P}$. A letter that names a plain number is thin and slanted: $c$, $d$, $t$. So $c\mathbf{x} + d\mathbf{y}$ says at a glance that $c$ and $d$ are numbers and $\mathbf{x}$ and $\mathbf{y}$ are vectors. On paper, where bold is hard, draw a little arrow over a vector letter.

**Scaling** a vector by a number is element-wise too: every slot gets multiplied by the number. It stretches the arrow. A negative number flips it around.

$$2(3, 4) = (6, 8) \qquad -1(3, 4) = (-3, -4)$$

$2\mathbf{a}$ means 2 copies of $\mathbf{a}$, read "two a". Python needs a star between the number and the vector, `2 * a`. On paper we leave the star out.

**Typed out.** $5(1, 2, 3)$ means 5 copies of the vector $(1, 2, 3)$, all added together:

$$5(1, 2, 3) = (1, 2, 3) + (1, 2, 3) + (1, 2, 3) + (1, 2, 3) + (1, 2, 3) = (5, 10, 15)$$

Look one slot at a time. The first slot is $1 + 1 + 1 + 1 + 1 = 5$, the second is $2 + 2 + 2 + 2 + 2 = 10$, and the third is $3 + 3 + 3 + 3 + 3 = 15$. So $5(1, 2, 3)$ has the same value as $(5, 10, 15)$. They are two ways to write the same vector, just like $5 + 5$ and $10$ are two ways to write the same number.

A few more, typed out the same way:

$$3(4, -1) = (4, -1) + (4, -1) + (4, -1) = (12, -3)$$

$-2(5, 1)$ is 2 copies of the opposite of $(5, 1)$, which is $(-5, -1)$:

$$-2(5, 1) = (-5, -1) + (-5, -1) = (-10, -2)$$

Zero copies is no move at all:

$$0(7, 2) = (0, 0)$$

Do these on paper first. Then check.

In [ ]:
a = np.array([3, 4])
b = np.array([1, -2])
print(a + b)
print(a - b)
print(2 * a)
print(-1 * a)
print(5 * np.array([1, 2, 3]))      # 5 copies of (1, 2, 3)

Adding means: walk along `a`, then walk along `b` from where you ended. `draw_vector` hands back the tip of the arrow, so the next arrow can start there.

In [ ]:
new_grid(8)
tip = draw_vector(a, label="a", label_at="middle")       # draw a, remember where it ended
draw_vector(b, start=tip, label="b", label_at="middle")  # start b where a ended
draw_vector(a + b, label="a + b")                        # the shortcut
show()

**Does the order matter?** Predict both lines, then run.


In [ ]:
print(a + b)
print(b + a)

**Your turn.** The cell below draws the same trip: `a` first, then `b`. Change it so that it walks `b` first and `a` second. Where do you end up? (This cell waits until you have changed it.)


In [ ]:
%%your_turn 0.2.1
new_grid(8)
tip = draw_vector(a, label="a", label_at="middle")       # walk a first ... (change me)
draw_vector(b, start=tip, label="b", label_at="middle")  # ... then b      (change me)
draw_vector(a + b, label="a + b")                        # the shortcut
show()


### Many routes, one place

Adding moves gives a new way to see a point. Start at home, $\mathbf{0} = (0, 0)$, and make the move $\mathbf{v} = (3, 4)$. Element-wise addition says where you land:

$$\mathbf{0} + \mathbf{v} = (0 + 3, 0 + 4) = (3, 4)$$

So the point $(3, 4)$ is simply where the move $(3, 4)$ takes you from home. It is the second reading from Part 0.1's "One list of numbers, two ways to read it", now written as a sum.

A trip can have more than two legs. Say you walk $\mathbf{u} = (4, 1)$, then $\mathbf{v} = (-1, 3)$, then $\mathbf{w} = (2, -2)$. You could walk the legs one at a time. You could first add $\mathbf{u}$ and $\mathbf{v}$ into one move, and then walk $\mathbf{w}$. Or you could walk $\mathbf{u}$, and then the one move $\mathbf{v} + \mathbf{w}$. The parentheses in the cell below say what is added first. Do all three routes end in the same place? Guess, run the cell, then run it again.


In [ ]:
u = np.array([4, 1])
v = np.array([-1, 3])
w = np.array([2, -2])
home = np.array([0, 0])

%guess_first ((home + u) + v) + w, (home + (u + v)) + w, home + (u + (v + w))


Now look at the three routes. Each one starts at home and walks its legs tip to tail.


In [ ]:
new_grid(6)
draw_path([u, v, w], labels=["u", "v", "w"])       # one leg at a time
draw_path([u + v, w], labels=["u + v", None])      # u and v added first (its w is the same leg)
draw_path([u, v + w], labels=[None, "v + w"])      # v and w added first (its u is the same leg)
show()


Every route lands on $(5, 2)$. Grouping the legs differently never changes where you end up: $(\mathbf{0} + \mathbf{u}) + \mathbf{v} = \mathbf{0} + (\mathbf{u} + \mathbf{v})$, and the same is true for any number of legs. Together with "the order does not matter", this is why a whole trip can be packed into one move, $\mathbf{u} + \mathbf{v} + \mathbf{w}$, with no parentheses at all.


**Does this look familiar?** You have seen this rule with ordinary numbers.

In [ ]:
print(2 * (a + b))
print(2 * a + 2 * b)

**Watch out!** Plain Python lists and NumPy arrays treat `+` differently.

In [ ]:
print([3, 4] + [1, -2])                        # plain lists get glued together
print(np.array([3, 4]) + np.array([1, -2]))    # NumPy arrays get added like vectors

A **velocity** is a vector that says how far something moves every second. If a car starts at the point $\mathbf{p}$ and drives with velocity $\mathbf{v}$, then after $t$ seconds it is at

$$\mathbf{p} + t \mathbf{v}$$

Predict all three lines.

In [ ]:
p = np.array([10, 0])      # where the car starts
v = np.array([3, 4])       # how far it moves each second
print(p + 1 * v)
print(p + 2 * v)
print(p + 5 * v)

### Mastery Coins 0.2

**Bronze (1 coin each)**

#### M 0.2.1

$(2, 5) + (4, 1)$

#### M 0.2.2

$(2, 5) - (4, 1)$

#### M 0.2.3

$3(2, -1)$

#### M 0.2.4

$-2(1, 4)$

#### M 0.2.5

$(1, 2, 3) + (10, 20, 30)$

**Silver (2 coins each)**

#### M 0.2.6

$2(1, 3) + 3(2, -1)$

#### M 0.2.7

A car starts at $(10, 0)$ and drives with velocity $(3, 4)$ meters every second. Where is it after 5 seconds?

#### M 0.2.8

In Roblox your character walks with velocity $(0, 0, 3)$ studs every frame. A conveyor belt also pushes it $(1, 0, 0)$ every frame. You start at $(0, 0, 0)$. Where are you after 4 frames?

**Gold (3 coins each)**

#### M 0.2.9

Find the vector $\mathbf{x}$ so that $(2, 5) + \mathbf{x} = (7, 1)$.

#### M 0.2.10

**(code)** Take $\mathbf{a} = (3, 4)$ and $\mathbf{b} = (1, -2)$. Print $2(\mathbf{a} + \mathbf{b})$ and $2\mathbf{a} + 2\mathbf{b}$. Try two more pairs of your own. Will the two always match? Say why.

#### M 0.2.11

What does $-5\mathbf{x}$ mean? It could mean $(-5)\mathbf{x}$: scale by the number $-5$. It could mean $-(5\mathbf{x})$: make 5 copies, then take the opposite. Or it could mean $5(-\mathbf{x})$: take the opposite $-\mathbf{x}$ first, then make 5 copies. Take $\mathbf{x} = (3, -2)$ and work out all three. Are they the same vector? Explain why, one slot at a time. Then show that $-\mathbf{x}$ is the same as $(-1)\mathbf{x}$, and find $\mathbf{x} + (-\mathbf{x})$. What kind of move is it?

In [ ]:
# Scratch space for the (code) coins in Part 0.2

---

## Part 0.3: Length and length-1 vectors

The **length** of a vector comes from the Pythagorean theorem. Square each number, add, and take the square root.

$$\lVert (3, 4) \rVert = \sqrt{3^2 + 4^2} = \sqrt{25} = 5$$

With three numbers there are three squares to add: $\lVert (1, 2, 2) \rVert = \sqrt{1 + 4 + 4} = 3$.

NumPy calls the length the **norm**. Roblox Studio calls it the `Magnitude`.

**Where the Pythagorean theorem comes in.** Walk to $(3, 4)$ the long way: first 3 steps east along the bottom, then 4 steps north. Together with the arrow $(3, 4)$, those two walks make a **right triangle**.

Look at its sides. The bottom of the triangle is just the x-coordinate, 3. The height of the triangle is just the y-coordinate, 4. The arrow itself is the long side across from the right angle, the **hypotenuse**. The Pythagorean theorem says: the long side squared equals the two short sides squared, added up.

$$3^2 + 4^2 = 9 + 16 = 25 = 5^2$$

So the arrow is 5 long. Predict where each side of the triangle will be, then run.

In [ ]:
new_grid(6)
corner = draw_vector(np.array([3, 0]))                  # the bottom: the x-coordinate
draw_vector(np.array([0, 4]), start=corner)             # the height: the y-coordinate
draw_vector(np.array([3, 4]))                           # the long side: the length
plt.text(1.5, -0.7, "x = 3", ha="center", fontsize=13, fontweight="bold")      # name each side
plt.text(3.25, 2, "y = 4", ha="left", va="center", fontsize=13, fontweight="bold")
plt.text(1.2, 2.5, "length 5", ha="right", fontsize=13, fontweight="bold")
show()

In [ ]:
a = np.array([3, 4])
print(np.sqrt(3**2 + 4**2))     # by hand
print(np.linalg.norm(a))        # NumPy's shortcut

Divide a vector by its own length and you get a vector of length 1 that points the same way. We write it with a hat.

$$\hat{\mathbf{a}} = \frac{\mathbf{a}}{\lVert \mathbf{a} \rVert}$$

The hat version keeps the **direction** and throws the **size** away. Roblox Studio calls it `a.Unit`. Predict what `a_hat` will be.

In [ ]:
a_hat = a / np.linalg.norm(a)
print(a_hat)
print(np.linalg.norm(a_hat))

new_grid(6)
draw_vector(a, label="a")
draw_vector(a_hat, label="a_hat")
show()

In Part 0.1 a car driving along a move was *heading in the direction of* that move. Now that vectors have a length, that direction gets a vector of its own: **the heading** is the length-1 vector that points the same way. A car's velocity is its **speed** times its heading. The speed is a number that says how fast. The heading is a length-1 vector that says which way.

Predict the speed and the heading, then run.


In [ ]:
v = np.array([30, 40])               # a car's velocity
speed = np.linalg.norm(v)
heading = v / speed
print(speed)
print(heading)
print(speed * heading)               # put them back together

**Your turn.** Find the length of $(6, 8)$ on paper. Then check it here, and make its length-1 version.

In [ ]:
c = np.array([6, 8])
print(np.linalg.norm(c))

### Mastery Coins 0.3

**Bronze (1 coin each)**

#### M 0.3.1

Find $\lVert (6, 8) \rVert$.

#### M 0.3.2

Find $\lVert (5, 12) \rVert$.

#### M 0.3.3

Find $\lVert (-3, 4) \rVert$.

#### M 0.3.4

Find $\lVert (2, 3, 6) \rVert$.

#### M 0.3.5

Write the length-1 version of $(0, -7)$.

**Silver (2 coins each)**

#### M 0.3.6

Write the length-1 version of $(8, 6)$. Check that its length really is 1.

#### M 0.3.7

In Roblox you stand at $(10, 0, 5)$ and an NPC stands at $(22, 0, 14)$. How many studs away is the NPC?

#### M 0.3.8

A car's velocity is $(30, 40)$ kilometers per hour. What is its speed? What is its heading, as a length-1 vector?

**Gold (3 coins each)**

#### M 0.3.9

A chaser NPC moves 4 studs every frame, straight toward a target. The move from the chaser to the target is $(12, 0, 9)$. Write the chaser's velocity.

#### M 0.3.10

You know $\lVert (3, 4) \rVert = 5$. Without using the Pythagorean theorem again, find $\lVert (30, 40) \rVert$, $\lVert (-6, -8) \rVert$ and $\lVert (1.5, 2) \rVert$. Write the rule you used.

In [ ]:
# Scratch space for Part 0.3

### Beyond the coins: other ways to measure size

The length $\lVert \mathbf{v} \rVert$ is not the only way to say how big a vector is. There is a whole family of them, called the **$L_p$ norms** (say "L-p norms"). Pick a number $p$ that is at least 1. Then

$$\lVert \mathbf{v} \rVert_p = \left( \lvert v_1 \rvert^p + \lvert v_2 \rvert^p + \dots + \lvert v_n \rvert^p \right)^{1/p}$$

In words: make every slot positive, raise each one to the power $p$, add them up, and then take the **$p$-th root**. Some new pieces here:

- $v_1$ is the first slot of $\mathbf{v}$, $v_2$ the second, and $n$ is the number of slots.
- $\lvert -3 \rvert = 3$. The bars make a number positive. This is called its **absolute value**.
- The power $1/p$ means the $p$-th root, which undoes "to the power $p$". The cube root of 8 is 2, because $2^3 = 8$.

Here are some of them for $\mathbf{v} = (3, 4)$:

| Name | The rule | For $(3, 4)$ | A picture |
| --- | --- | --- | --- |
| $L_1$ | add up the sizes of the slots | $3 + 4 = 7$ | a taxi driving on city streets, counting blocks |
| $L_2$ | square, add, square root | $\sqrt{25} = 5$ | the length from Part 0.3, as the crow flies |
| $L_3$ | cube, add, cube root | $\sqrt[3]{91} \approx 4.50$ | in between |
| $L_\infty$ | the biggest size of any slot | $4$ | a chess king: how many king moves to get there |

$L_2$ is the length you already know, so $\lVert \mathbf{v} \rVert$ with no number means $\lVert \mathbf{v} \rVert_2$.

Two special members:

- **$L_\infty$** (say "L-infinity"): let $p$ grow bigger and bigger. The biggest slot wins more and more, until only it counts. So $\lVert \mathbf{v} \rVert_\infty$ is the biggest slot size: $\lVert (3, 4) \rVert_\infty = 4$, and $\lVert (-7, 2) \rVert_\infty = 7$.
- **$L_0$** (say "L-zero"): counts how many slots are not zero. $\lVert (3, 4) \rVert_0 = 2$ and $\lVert (0, 5, 0) \rVert_0 = 1$. Many people call it a norm, but it is not a real one: Deep 8 at the bottom of this notebook shows why.

**Two facts.** A fact that is always true is called a **theorem**. For any vector $\mathbf{v}$:

1. **A bigger $p$ never gives a bigger size.** $\lVert \mathbf{v} \rVert_\infty \le \lVert \mathbf{v} \rVert_2 \le \lVert \mathbf{v} \rVert_1$. For $(3, 4)$: $4 \le 5 \le 7$. The same goes for any two: if $p \le q$, then $\lVert \mathbf{v} \rVert_q \le \lVert \mathbf{v} \rVert_p$.
2. **One slot, one answer.** If only one slot is not zero, every $L_p$ size is the same: $\lVert (0, 5) \rVert_p = 5$ for every $p$.

There are more theorems about these sizes in Deep 8, at the bottom of this notebook.

**An $L_p$ function in Python.** NumPy does each step to every slot at once. `np.abs` makes every slot positive, `**` raises every slot to a power, and `np.sum` adds the slots up. `def` makes a new command of your own, called a **function**, and `return` hands back its answer. Predict every number, then run.

In [ ]:
def lp_norm(v, p):
    sizes = np.abs(v).astype(float)          # every slot positive (as a decimal, so big powers fit)
    return np.sum(sizes ** p) ** (1 / p)     # power p, add up, then the p-th root


def max_norm(v):                             # L-infinity: the biggest slot size
    return np.max(np.abs(v))


def count_norm(v):                           # L-0: how many slots are not zero
    return np.count_nonzero(v)


v = np.array([3, 4])
print(lp_norm(v, 1), lp_norm(v, 2), lp_norm(v, 3), max_norm(v), count_norm(v))
print(lp_norm(v, 100))                       # a huge p: which one is it close to?
print(np.linalg.norm(v, 1), np.linalg.norm(v, np.inf))   # NumPy's own versions, to compare

**Your turn.** Use your functions. Python can test a theorem for you: `a <= b` prints `True` or `False`.

1. On paper, find $L_1$, $L_2$, $L_\infty$ and $L_0$ of $(-2, 6)$ and of $(1, 1, 1, 1)$. Then check with your functions.
2. Test theorem 1 on three vectors of your own: one with a negative slot, and one with 3 or more slots.
3. Test theorem 2 on $(0, -7, 0)$ for $p = 1$, $2$, $3$ and $L_\infty$.

In [ ]:
v = np.array([-2, 6])
print(max_norm(v) <= lp_norm(v, 2) <= lp_norm(v, 1))        # theorem 1 for (-2, 6): True or False?

# your tests here

---

## Part 0.4: The inner product: sign and size

To find the inner product $\langle \mathbf{a}, \mathbf{b} \rangle$, multiply the matching slots and add the results.

$$\langle (3, 4), (4, -3) \rangle = (3)(4) + (4)(-3) = 12 - 12 = 0$$

The answer is **one number**. It is not a vector.

Many books call it the **dot product**. That is why NumPy's shortcut is `np.dot(a, b)` and Roblox's is `a:Dot(b)`. It is the same thing with two names.

In [ ]:
p = np.array([3, 4])
q = np.array([4, -3])
print(p[0] * q[0] + p[1] * q[1])    # by hand
print(np.dot(p, q))                 # NumPy's shortcut

The inner product came out as zero. Look at what that means.

In [ ]:
new_grid(6)
draw_vector(p, label="p")
draw_vector(q, label="q")
show()
print("angle:", angle_between(p, q), "degrees")

### The angle gallery

Here are five vectors. **Before you run the second cell**, write down on paper:

1. Which pairs look perpendicular ($90^\circ$)?
2. Which pair has the biggest angle between them?
3. Your best guess for the angle between `a` and `b`.

In [ ]:
a = np.array([4, 0])
b = np.array([3, 3])
c = np.array([0, 5])
d = np.array([-4, 2])
e = np.array([-2, -4])

new_grid(6)
draw_vector(a, label="a")
draw_vector(b, label="b")
draw_vector(c, label="c")
draw_vector(d, label="d")
draw_vector(e, label="e")
show()

In [ ]:
print("a and b:  inner product =", np.dot(a, b), "  angle =", angle_between(a, b))
print("a and c:  inner product =", np.dot(a, c), "  angle =", angle_between(a, c))
print("a and d:  inner product =", np.dot(a, d), "  angle =", angle_between(a, d))
print("a and e:  inner product =", np.dot(a, e), "  angle =", angle_between(a, e))
print("d and e:  inner product =", np.dot(d, e), "  angle =", angle_between(d, e))

**Look for a pattern.** When is the inner product positive? Zero? Negative? Write your rule in one sentence before you read on.

**Your turn.** Add lines for two more pairs. Guess each angle before you run it.

In [ ]:
print("b and c:  inner product =", np.dot(b, c), "  angle =", angle_between(b, c))

### The sign rule

| $\langle \mathbf{a}, \mathbf{b} \rangle$ is | so the angle between $\mathbf{a}$ and $\mathbf{b}$ is |
| --- | --- |
| positive | less than $90^\circ$ |
| zero | exactly $90^\circ$ |
| negative | more than $90^\circ$ |

Zero is the special one. A zero inner product means **perpendicular**, with no square roots and no protractor.

### What about the size?

The sign tells you the kind of angle. What changes the **size** of the number? Predict each line.

In [ ]:
a = np.array([3, 4])
b = np.array([2, 1])
print(np.dot(a, b))
print(np.dot(2 * a, b))       # make a twice as long
print(np.dot(a, 3 * b))       # make b three times as long
print(np.dot(-1 * a, b))      # flip a around

One more surprise. Take the inner product of a vector with itself.

In [ ]:
print(np.dot(a, a))
print(np.linalg.norm(a) ** 2)

The length was hiding inside the inner product all along:

$$\langle \mathbf{a}, \mathbf{a} \rangle = \lVert \mathbf{a} \rVert^2$$

### Mastery Coins 0.4

**Bronze (1 coin each)**

#### M 0.4.1

Find $\langle (2, 3), (4, 1) \rangle$.

#### M 0.4.2

Find $\langle (1, 5), (5, -1) \rangle$. What does your answer tell you?

#### M 0.4.3

Find $\langle (2, 1), (-3, 1) \rangle$. Is the angle less than or more than $90^\circ$?

#### M 0.4.4

Find $\langle (1, 2, 3), (4, 0, -1) \rangle$.

#### M 0.4.5

Find $\langle (6, 8), (6, 8) \rangle$. What is its square root, and what does that number mean?

**Silver (2 coins each)**

#### M 0.4.6

A car points along $\mathbf{h} = (3, 4)$. The move from the car to the gas station is $\mathbf{d} = (5, -10)$. Find $\langle \mathbf{h}, \mathbf{d} \rangle$. Is the station ahead of the car or behind it?

#### M 0.4.7

You pull a sled with force $(40, 30, 0)$ newtons. It slides $(10, 0, 0)$ meters. The work you do is $\langle \text{force}, \text{move} \rangle$. How much work is that? Which part of your pull was wasted?

#### M 0.4.8

Take $\mathbf{a} = (3, 4)$ and $\mathbf{b} = (2, 1)$, so $\langle \mathbf{a}, \mathbf{b} \rangle = 10$. Without multiplying everything out again, find $\langle 2\mathbf{a}, \mathbf{b} \rangle$, $\langle \mathbf{a}, 3\mathbf{b} \rangle$, $\langle 2\mathbf{a}, 3\mathbf{b} \rangle$ and $\langle -\mathbf{a}, \mathbf{b} \rangle$. What changed the size? What changed the sign?

**Gold (3 coins each)**

#### M 0.4.9

Find the number $t$ that makes $(3, t)$ perpendicular to $(4, 6)$.

#### M 0.4.10

The car from M 0.4.6 points along $\mathbf{h} = (3, 4)$. Find a vector that points to the car's **left**. Use it and an inner product to decide whether the gas station at $\mathbf{d} = (5, -10)$ is on the left or the right. Now you know two things about where the station is. Say them.

In [ ]:
# Scratch space for Part 0.4

---

## Part 0.5: Raw or normalized? How alike are two vectors?

The plain inner product is called the **raw** inner product. It mixes two things together: how **long** the vectors are, and how well they **line up**.

To keep only the lining-up, divide by both lengths. That is the **normalized** inner product.

$$\frac{\langle \mathbf{a}, \mathbf{b} \rangle}{\lVert \mathbf{a} \rVert \, \lVert \mathbf{b} \rVert}$$

It always lands between $-1$ and $1$.

| normalized inner product | what it means |
| --- | --- |
| $1$ | same direction |
| $0$ | perpendicular |
| $-1$ | opposite directions |

In your ML class this number is called **cosine similarity**.

Compare $\mathbf{a} = (3, 4)$ with four other vectors. First draw them. Which one looks most like $\mathbf{a}$?

In [ ]:
a = np.array([3, 4])
b = np.array([6, 8])
c = np.array([24, 7])
d = np.array([-8, 6])
e = np.array([-6, -8])

new_grid(30)
draw_vector(a, label="a", label_at="middle")
draw_vector(b, label="b")
draw_vector(c, label="c")
draw_vector(d, label="d")
draw_vector(e, label="e")
show()

Now the raw inner products. Which vector do **they** say is most like $\mathbf{a}$?

In [ ]:
print("raw a and b:", np.dot(a, b))
print("raw a and c:", np.dot(a, c))
print("raw a and d:", np.dot(a, d))
print("raw a and e:", np.dot(a, e))

Now the normalized ones. Predict each before you run. Did the winner change? Why?

In [ ]:
print("normalized a and b:", np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))
print("normalized a and c:", np.dot(a, c) / (np.linalg.norm(a) * np.linalg.norm(c)))
print("normalized a and d:", np.dot(a, d) / (np.linalg.norm(a) * np.linalg.norm(d)))
print("normalized a and e:", np.dot(a, e) / (np.linalg.norm(a) * np.linalg.norm(e)))

### Two routes to the same number

- **Route A:** normalize both vectors first, then take the inner product: $\langle \hat{\mathbf{a}}, \hat{\mathbf{b}} \rangle$.
- **Route B:** take the inner product first, then divide by both lengths.

Do both routes on paper for $(3, 4)$ and $(8, 6)$. Then check.

In [ ]:
a = np.array([3, 4])
b = np.array([8, 6])

a_hat = a / np.linalg.norm(a)
b_hat = b / np.linalg.norm(b)
print("Route A:", np.dot(a_hat, b_hat))

print("Route B:", np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

### Look inside the library

`angle_between` turns the normalized inner product into degrees. It is not magic. Somebody wrote it, and you can read it. Find the inner product. Find the two lengths. Which route does it use?

In [ ]:
import inspect
print(inspect.getsource(angle_between))

### Mastery Coins 0.5

**Bronze (1 coin each)**

#### M 0.5.1

Find the normalized inner product of $(3, 4)$ and $(6, 8)$. What does it tell you?

#### M 0.5.2

Find the normalized inner product of $(3, 4)$ and $(-8, 6)$. What does it tell you?

#### M 0.5.3

Find the normalized inner product of $(3, 4)$ and $(-6, -8)$. What does it tell you?

#### M 0.5.4

Find the normalized inner product of $(3, 4)$ and $(24, 7)$.

#### M 0.5.5

True or false: a normalized inner product can equal 3.

**Silver (2 coins each)**

#### M 0.5.6

Use Route A and then Route B on $(20, 15)$ and $(7, 24)$. Show that both give the same number.

#### M 0.5.7

Game characters have stats (strength, stealth, armor, magic, luck). Knight $= (9, 3, 8, 1, 5)$, Rogue $= (4, 9, 3, 2, 7)$, Mage $= (2, 5, 1, 9, 6)$. Find the raw inner product for each of the three pairs. Which pair is biggest?

#### M 0.5.8

**(code)** Find the normalized inner product for each pair of Knight, Rogue and Mage. Which two play most alike now? Why did the answer change?

**Gold (3 coins each)**

#### M 0.5.9

Take $(5, 12)$ and $(12, 5)$. Use Route B to get the exact answer as a fraction, then as a decimal. Now use Route A, but round every number in the two hat vectors to 2 decimal places before you multiply. The two answers do not match. Which one is right? When is the best time to round?

#### M 0.5.10

Two runners are written as (distance, time): $(400 \text{ m}, 60 \text{ s})$ and $(800 \text{ m}, 130 \text{ s})$. A friend finds the angle between the two vectors. Then she switches to kilometers and minutes and gets a different angle. Which angle is right? What should she do first?

In [ ]:
# Scratch space for the (code) coins in Part 0.5
knight = np.array([9, 3, 8, 1, 5])
rogue = np.array([4, 9, 3, 2, 7])
mage = np.array([2, 5, 1, 9, 6])

---

## Part 0.6: Roblox: toward, away and in view

Your character is at a point $\mathbf{P}$. An NPC is at a point $\mathbf{N}$. Three vectors matter.

- $\mathbf{d} = \mathbf{N} - \mathbf{P}$ is the move from you to the NPC.
- $\mathbf{v}$ is your velocity: which way you are **walking**, and how fast.
- $\mathbf{f}$ is your facing: which way you are **looking**. It has length 1. In Roblox Studio it is called the `LookVector`.

Walking and looking are two different things. You can walk backwards!

| Question | Work out | How to read the answer |
| --- | --- | --- |
| Am I getting closer? | $\langle \mathbf{v}, \mathbf{d} \rangle$ | positive: closer. negative: farther. zero: neither. |
| How fast am I closing in? | $\langle \mathbf{v}, \hat{\mathbf{d}} \rangle$ | studs per second toward the NPC |
| Can I see the NPC? | $\langle \mathbf{f}, \hat{\mathbf{d}} \rangle$ | $0.5$ or more: yes, it is in view |

The first question only needs the **sign**, so the raw inner product is enough. The other two need a real **size**, so $\mathbf{d}$ has to be normalized first.

Why $0.5$? Your character sees $60^\circ$ to each side, and $0.5$ is the normalized inner product of two vectors that are $60^\circ$ apart.

In [ ]:
P = np.array([0, 0, 0])        # you
N = np.array([6, 0, 8])        # the NPC
d = N - P
print(d)
print(np.linalg.norm(d))       # how far away is the NPC?

Here is the view from above. `P[[0, 2]]` keeps slot 0 and slot 2 of `P`, which are $x$ and $z$.

Before you run it: is `v` pointing toward the NPC? Is `f`?

In [ ]:
v = np.array([3, 0, 4])        # walking
f = np.array([1, 0, 0])        # looking

ax = new_grid(10)
ax.set_xlabel("x")
ax.set_ylabel("z")
draw_point(P[[0, 2]], "you")
draw_point(N[[0, 2]], "NPC")
draw_vector(v[[0, 2]], start=P[[0, 2]], label="v")         # walking
draw_vector(3 * f[[0, 2]], start=P[[0, 2]], label="f")     # looking (drawn 3 long so you can see it)
show()

**Am I getting closer?** Only the sign matters.

In [ ]:
print(np.dot(v, d))

**How fast am I closing in?** Normalize $\mathbf{d}$, then take the inner product with $\mathbf{v}$. Compare it with your full speed.

In [ ]:
d_hat = d / np.linalg.norm(d)
print(d_hat)
print(np.dot(v, d_hat))        # closing speed
print(np.linalg.norm(v))       # your full speed

**Can I see the NPC?**

In [ ]:
print(np.dot(f, d_hat))        # 0.5 or more means "in view"
print(angle_between(f, d))     # the same news, in degrees

**Your turn.** Go back to the cell where `v` and `f` are made.

1. Change `v` so that you walk straight away from the NPC. Predict all three answers, then run the cells again.
2. Change `f` so that you look straight at the NPC. What is the biggest the view number can ever be?
3. Make your character walk toward the NPC while looking away from it.

### Mastery Coins 0.6

For coins M 0.6.1 to M 0.6.8, you are at $\mathbf{P} = (0, 0, 0)$ and the NPC stands at $\mathbf{N} = (6, 0, 8)$.

**Bronze (1 coin each)**

#### M 0.6.1

Write $\mathbf{d} = \mathbf{N} - \mathbf{P}$. How many studs away is the NPC?

#### M 0.6.2

You walk with $\mathbf{v} = (3, 0, 4)$. Find $\langle \mathbf{v}, \mathbf{d} \rangle$. Are you getting closer or farther?

#### M 0.6.3

Now you walk with $\mathbf{v} = (-4, 0, 3)$. Find $\langle \mathbf{v}, \mathbf{d} \rangle$. Closer, farther, or neither? Describe how you are moving.

#### M 0.6.4

Now you walk with $\mathbf{v} = (0, 0, -5)$. Find $\langle \mathbf{v}, \mathbf{d} \rangle$. Closer or farther?

#### M 0.6.5

You look along $\mathbf{f} = (1, 0, 0)$. Find $\langle \mathbf{f}, \mathbf{d} \rangle$. Is the NPC somewhere in front of you, or behind you?

**Silver (2 coins each)**

#### M 0.6.6

Here $\hat{\mathbf{d}} = (0.6, 0, 0.8)$. Find the closing speed $\langle \mathbf{v}, \hat{\mathbf{d}} \rangle$ for $\mathbf{v} = (3, 0, 4)$ and for $\mathbf{v} = (5, 0, 0)$. Both walkers have speed 5. Why are the closing speeds different?

#### M 0.6.7

The NPC is in view when $\langle \mathbf{f}, \hat{\mathbf{d}} \rangle$ is $0.5$ or more. Is it in view when $\mathbf{f} = (1, 0, 0)$? When $\mathbf{f} = (0, 0, 1)$? When $\mathbf{f} = (0, 0, -1)$?

#### M 0.6.8

You walk with $\mathbf{v} = (3, 0, 4)$ while looking along $\mathbf{f} = (-0.6, 0, -0.8)$. Are you getting closer? Is the NPC in view? What is your character doing?

**Gold (3 coins each)**

#### M 0.6.9

Now the NPC can fly. You stay at $(0, 0, 0)$ and look along $\mathbf{f} = (0, 0, 1)$. The NPC hovers at $(2, 3, 6)$. How far away is it? Is it in view? Then it climbs to $(2, 6, 3)$. How far away is it now? Is it in view? What changed, and what did not?

#### M 0.6.10

**(code)** Now you program the NPC. A guard stands at $(0, 0, 0)$ and looks along $\mathbf{f} = (0, 0, 1)$. Three players stand at $\mathbf{A} = (3, 0, 4)$, $\mathbf{B} = (8, 0, -6)$ and $\mathbf{C} = (-12, 0, 5)$. For each player, print the view number $\langle \mathbf{f}, \hat{\mathbf{d}} \rangle$. Which players can the guard see? Which player is the guard looking at most directly?

In [ ]:
# Scratch space for the (code) coins in Part 0.6
guard = np.array([0, 0, 0])
f = np.array([0, 0, 1])
A = np.array([3, 0, 4])
B = np.array([8, 0, -6])
C = np.array([-12, 0, 5])

---

## Cheat sheet: three ways to say the same thing

| Idea | On paper | NumPy | Roblox Studio |
| --- | --- | --- | --- |
| the length of $\mathbf{a}$ | $\lVert \mathbf{a} \rVert$ | `np.linalg.norm(a)` | `a.Magnitude` |
| the length-1 version of $\mathbf{a}$ | $\hat{\mathbf{a}}$ | `a / np.linalg.norm(a)` | `a.Unit` |
| the inner product of $\mathbf{a}$ and $\mathbf{b}$ | $\langle \mathbf{a}, \mathbf{b} \rangle$ | `np.dot(a, b)` | `a:Dot(b)` |

The inner product is also called the **dot product**. It is the same thing with two names.

---

## Proficiency check

Do all six on paper, with no notes. Then show your teacher.

1. $(2, -1) + (1, 4)$
2. $3(2, -1)$
3. A car's velocity is $(5, 12)$ meters per second. What is its speed?
4. Find $\langle (2, -1), (1, 2) \rangle$. What does the answer tell you?
5. Write the length-1 version of $(0, 5)$.
6. In Roblox you are at $(0, 0, 0)$ and an NPC is at $(6, 0, 8)$. You walk with $\mathbf{v} = (-3, 0, -4)$. Are you getting closer or farther? Show the inner product.

**Notebook task.** In the empty cell below, make two vectors of your own, draw both with labels, and print the angle between them.

---

# The Deep End

You do not have to do these. They are longer, they have fewer hints, and some of them point at things we have not learned yet. Pick one that looks interesting.

## Deep 1: The stretch test

If you make a vector 10 times longer, does the angle to another vector change? Predict, then test. Try it with a negative number too.

Then explain this: the inner product of two **length-1** vectors is a number between $-1$ and $1$. What does that number tell you?

In [ ]:
a = np.array([3, 1])
b = np.array([1, 2])
print(angle_between(a, b))
print(angle_between(10 * a, b))

a_hat = a / np.linalg.norm(a)
b_hat = b / np.linalg.norm(b)
print(np.dot(a_hat, b_hat))

## Deep 2: The units trap

Two runners. Each vector is (distance, time).

Find the angle between them when distance is in meters and time is in seconds. Then change to kilometers and minutes. **Nothing about the runners changed.** Did the angle change? Why? What would you do about it?

*This leads to: feature scaling in your ML class.*

In [ ]:
runner1 = np.array([400, 60])      # meters, seconds
runner2 = np.array([800, 130])
print(angle_between(runner1, runner2))

runner1_km_min = np.array([0.4, 1.0])            # kilometers, minutes
runner2_km_min = np.array([0.8, 130 / 60])
print(angle_between(runner1_km_min, runner2_km_min))

## Deep 3: The perpendicular maker

For $\mathbf{v} = (3, 1)$, find a vector whose inner product with $\mathbf{v}$ is zero. Now do it for $(2, 5)$, then $(-4, 7)$. Find a rule that works for **any** vector $(x, y)$, and draw a few to check.

Your rule gives a car's left or its right. Which one? How would you get the other?

*This leads to: rotations (Module 7).*

In [ ]:
v = np.array([3, 1])
w = np.array([0, 0])     # change me so that the inner product is 0
print(np.dot(v, w))

## Deep 4: Is the shortcut ever longer?

Walking $\mathbf{a}$ and then $\mathbf{b}$ gets you to the same place as walking $\mathbf{a} + \mathbf{b}$. Can the shortcut ever be **longer** than the two legs put together? When is it exactly the same length?

The cell below tries 1000 random pairs. It uses a `for` loop, which we will learn next time. Read it and guess what each line does.

In [ ]:
rng = np.random.default_rng()
worst = 0
for i in range(1000):
    a = rng.integers(-9, 10, size=2)       # a random vector with whole numbers
    b = rng.integers(-9, 10, size=2)
    shortcut = np.linalg.norm(a + b)
    two_legs = np.linalg.norm(a) + np.linalg.norm(b)
    if shortcut > two_legs + 0.000001:
        worst = worst + 1
print("times the shortcut was longer:", worst)

## Deep 5: Computers cannot do fractions perfectly

Predict what this prints. Then find out why. (`np.isclose` is how we ask "are these close enough to be equal?")

In [ ]:
print(0.1 + 0.2)
print(0.1 + 0.2 == 0.3)
print(np.isclose(0.1 + 0.2, 0.3))

## Deep 6: Change the library

Run `print(inspect.getsource(vecdraw))` and find the list called `COLORS`. Change the first color by running `COLORS[0] = "black"` in a new cell, and redraw the angle gallery. To put the library back the way it was, run the import cell again.

Then write your own function `draw_triangle(a, b)` that draws `a`, then `b` from the tip of `a`, then the shortcut `a + b`.

In [ ]:
def draw_triangle(a, b):
    tip = draw_vector(a, label="a")
    # finish me


new_grid(8)
draw_triangle(np.array([3, 4]), np.array([1, -2]))
show()

## Deep 7: The chase

An NPC chases you. On every step it works out the move from itself to you, normalizes it, and walks along it at its own speed. You just keep walking east.

Read the loop and guess what each line does. Then run it.

1. Does the NPC catch you? On which step?
2. Change `npc_speed` to 1. Does it ever catch you? Why not?
3. You walk at speed 1. What is the slowest NPC that can still catch you?
4. Give the NPC a field of view: it only chases while it can see you.

*This leads to: linear combinations (Module 1), where you will work out exactly where everyone ends up.*

In [ ]:
me = np.array([0.0, 0.0, 0.0])
my_velocity = np.array([1.0, 0.0, 0.0])

npc = np.array([20.0, 0.0, 15.0])
npc_speed = 2.0

for step in range(60):
    d = me - npc                              # the move from the NPC to you
    distance = np.linalg.norm(d)
    if distance < npc_speed:
        print("Caught on step", step)
        break
    npc = npc + npc_speed * d / distance      # one step straight toward you
    me = me + my_velocity                     # you keep walking

print("you:", me)
print("NPC:", np.round(npc, 2))

## Deep 8: Other sizes, same rules?

This one uses `lp_norm`, `max_norm` and `count_norm` from "Beyond the coins" in Part 0.3. Run that cell first. To test whether two decimals are equal, use `np.isclose(a, b)`, not `==` (Deep 5 says why).

Three more theorems. For any vector $\mathbf{v}$ with $n$ slots, any number $c$, and any vectors $\mathbf{a}$ and $\mathbf{b}$:

1. **Never too much smaller.** $\lVert \mathbf{v} \rVert_1 \le \sqrt{n} \, \lVert \mathbf{v} \rVert_2$ and $\lVert \mathbf{v} \rVert_2 \le \sqrt{n} \, \lVert \mathbf{v} \rVert_\infty$ and $\lVert \mathbf{v} \rVert_1 \le n \lVert \mathbf{v} \rVert_\infty$. For $(3, 4)$, $n = 2$: $7 \le 7.07$, $5 \le 5.66$ and $7 \le 8$.
2. **Scaling scales the size.** $\lVert c\mathbf{v} \rVert_p = \lvert c \rvert \, \lVert \mathbf{v} \rVert_p$, for every $p$ from 1 up and for $L_\infty$. Making a vector 3 times as long makes every size 3 times as big, and flipping it changes nothing.
3. **The shortcut is never longer.** $\lVert \mathbf{a} + \mathbf{b} \rVert_p \le \lVert \mathbf{a} \rVert_p + \lVert \mathbf{b} \rVert_p$, for every $p$ from 1 up and for $L_\infty$. Deep 4 tested this for the ordinary length.

Try these.

1. Theorem 1 can be an exact tie. Find a vector with $\lVert \mathbf{v} \rVert_1 = \sqrt{n} \, \lVert \mathbf{v} \rVert_2$. What is special about its slots?
2. Test theorem 2 with $c = -3$ and $\mathbf{v} = (3, 4)$, for $p = 1$, $2$ and $3$ and for $L_\infty$. Then show that $L_0$ breaks it.
3. Test theorem 3 with $\mathbf{a} = (3, 0)$ and $\mathbf{b} = (0, 4)$ for $L_1$, $L_2$ and $L_\infty$. Which one is a tie?
4. **Challenge.** Try `lp_norm` with $p = 0.5$, which is less than 1, on $\mathbf{a} = (1, 0)$ and $\mathbf{b} = (0, 1)$. Is the shortcut $\mathbf{a} + \mathbf{b}$ longer than going along $\mathbf{a}$ and then $\mathbf{b}$? What does that say about $p = 0.5$?

In [ ]:
v = np.array([3, 4])
print(np.isclose(lp_norm(-3 * v, 3), 3 * lp_norm(v, 3)))   # theorem 2 with c = -3 and p = 3
print(count_norm(-3 * v), count_norm(v))                   # and for L-0?

# your tests here

## Deep 9: Write the chaser's brain

In Deep 7 the NPC chased you. Now **you** are the chaser, and you write the rule.

The NPC runs laps around a track: the unit square with corners $(1, 1)$, $(2, 1)$, $(2, 2)$ and $(1, 2)$. It runs at a steady speed, counterclockwise. You start at home, $(0, 0)$, where the **base station** is. Every step, the base station's radar says where the NPC is. That **radar echo** is measured from the base station, not from you. So you have to work out the move from you to the NPC yourself.

Your rule hands back two vectors:

- `go`, the way you move: you are heading in the direction of `go`. With length 1 you move at full speed, a shorter `go` moves you more slowly, and $(0, 0)$ means "stand still". A `go` longer than 1 is cut back to length 1, because that is your top speed.
- `facing`, the way you look, a length-1 vector. Facing the NPC while you chase it is smart, but you do not have to move the way you are facing.

Time goes forward in small jumps of `step_time` seconds. That is how often the radar answers, how often your rule runs, and how often the paths get a new point. You **catch** the NPC if, at any moment during a step, you come inside its **touch zone**. The zone can be a circle around the NPC (`npc_shape = "circle"`) or a rectangular box (`npc_shape = "box"`). The chase stops when you catch it, when time runs out, or, as a safety limit, when you get more than `too_far` away.

Run the next cell first. It sets up the track and the chase. You do not have to read all of it, but the comments say what each part does. Two functions in it, `closest_during_step` and `caught_in_box`, decide whether you touched the NPC. Treat them as black boxes for now: in Deep 10 you open them and build them yourself.

In [ ]:
step_time = 0.1      # seconds between radar echoes: how finely time is chopped up
npc_shape = "circle" # the NPC's touch zone: "circle" or "box"
touch_range = 0.1    # the circle's radius
touch_box = (0.15, 0.08)   # the box's half-width and half-height
npc_speed = 0.5      # how far the NPC runs every second
my_speed = 0.4       # how far you can move every second
max_time = 60        # seconds before the chase is called off
too_far = 100        # safety limit: stop if you get farther away than this


def npc_position(t):
    """Where the NPC is at time t, measured from the base station at (0, 0)."""
    along = (npc_speed * t) % 4                      # how far along this lap it is (one lap is 4 long)
    if along < 1:
        return np.array([2.0, 1.0 + along])          # up the right side
    if along < 2:
        return np.array([2.0 - (along - 1), 2.0])    # left along the top
    if along < 3:
        return np.array([1.0, 2.0 - (along - 2)])    # down the left side
    return np.array([1.0 + (along - 3), 1.0])        # right along the bottom


def closest_during_step(start_move, end_move):
    """The closest you and the NPC were, at the same moment, during one step.

    It is not enough for your path to cross the NPC's path: you both have to be
    there at the same time. start_move and end_move are the moves from you to
    the NPC at the start and at the end of the step. Both of you move in straight
    lines during the step, so that move changes along a straight line too. An
    inner product finds the moment in the step when it is shortest.
    """
    change = end_move - start_move
    if np.dot(change, change) == 0:
        return np.linalg.norm(start_move)
    s = -np.dot(start_move, change) / np.dot(change, change)   # how far into the step the closest moment is
    s = min(1, max(0, s))                                      # it has to be inside this step
    return np.linalg.norm(start_move + s * change)


def caught_in_box(start_move, end_move, half_width, half_height):
    """True if, at some moment during one step, you were inside the NPC's box.

    The box is centered on the NPC, half_width to each side and half_height up
    and down. Each slot of the move from you to the NPC gives two linear
    inequalities for the moment s; you were inside if some s from 0 to 1 fits them all.
    """
    change = end_move - start_move
    low = 0.0                                      # the moments still possible: from low ...
    high = 1.0                                     # ... to high
    for start, step, half in [(start_move[0], change[0], half_width), (start_move[1], change[1], half_height)]:
        if step == 0:                              # this slot never changes during the step
            if abs(start) > half:
                return False
        else:
            a = (-half - start) / step             # the moments this slot is at -half and at +half
            b = (half - start) / step
            low = max(low, min(a, b))
            high = min(high, max(a, b))
    return low <= high


def draw_chase(my_path, npc_path, looks):
    my_path = np.array(my_path)
    npc_path = np.array(npc_path)
    plt.figure(figsize=(6, 6))
    plt.plot([1, 2, 2, 1, 1], [1, 1, 2, 2, 1], color="gray", linestyle=":", label="track")
    plt.plot(npc_path[:, 0], npc_path[:, 1], color="#CC79A7", linestyle="--", linewidth=2, label="NPC")
    plt.plot(my_path[:, 0], my_path[:, 1], color="#009E73", linewidth=2, label="you, the chaser")
    end = npc_path[-1]
    if npc_shape == "circle":                                 # the NPC's touch zone, where it is at the end
        zone = plt.Circle(end, touch_range, fill=False, color="#CC79A7", linewidth=2)
    else:
        zone = plt.Rectangle((end[0] - touch_box[0], end[1] - touch_box[1]), 2 * touch_box[0], 2 * touch_box[1],
                             fill=False, color="#CC79A7", linewidth=2)
    plt.gca().add_patch(zone)
    for place, facing in looks[::10]:                         # every 10th step: which way you were facing
        plt.arrow(place[0], place[1], 0.15 * facing[0], 0.15 * facing[1], head_width=0.04, color="black")
    plt.plot(0, 0, "^", color="black", markersize=14)          # the radar base station, at (0, 0)
    for r in [0.2, 0.3, 0.4]:                                   # radar waves going out from it
        angles = np.linspace(0.2, 1.4, 20)
        plt.plot(r * np.cos(angles), r * np.sin(angles), color="black", linewidth=1)
    plt.text(0.05, -0.25, "radar base station (0, 0)")
    echo = npc_path[-1]                                         # the last radar echo: a vector from the base
    plt.annotate("", xy=echo, xytext=(0, 0), arrowprops={"arrowstyle": "->", "linestyle": "--", "color": "gray"})
    plt.text(echo[0] / 2 - 0.1, echo[1] / 2 + 0.05, "radar echo", color="gray", ha="right")
    plt.xlim(-0.5, 2.5)
    plt.ylim(-0.5, 2.5)
    plt.gca().set_aspect("equal")
    plt.grid(alpha=0.3)
    plt.legend(loc="upper left")
    show()


def chase(rule):
    me = np.array([0.0, 0.0])           # you start at the base station
    previous_echo = None                # there is no echo before the first one
    my_path = [me]
    npc_path = []
    looks = []                          # where you were and which way you faced, for the picture
    t = 0.0
    result = "time ran out"
    while t < max_time:
        echo = npc_position(t)          # the radar: where the NPC is, from the base station
        npc_path.append(echo)
        go, facing = rule(me, echo, previous_echo, t)
        if np.linalg.norm(go) > 1:                      # the speed limit: no go longer than 1
            go = go / np.linalg.norm(go)
        looks.append((me, facing))
        next_me = me + my_speed * step_time * go        # one step in the direction of go
        next_echo = npc_position(t + step_time)
        t = t + step_time
        my_path.append(next_me)
        if npc_shape == "circle":
            caught = closest_during_step(echo - me, next_echo - next_me) <= touch_range
        else:
            caught = caught_in_box(echo - me, next_echo - next_me, touch_box[0], touch_box[1])
        if caught:
            npc_path.append(next_echo)
            result = f"caught it at t = {t:.1f} seconds"
            break
        if np.linalg.norm(next_echo - next_me) > too_far:
            result = "too far away: stopped by the safety limit"
            break
        me = next_me
        previous_echo = echo
    draw_chase(my_path, npc_path, looks)
    print(result)

Here is a first rule: look straight at the NPC and walk straight at it. You are slower than the NPC ($0.4$ against $0.5$). **Predict:** will you catch it? Then run.

In [ ]:
def my_rule(me, radar_echo, previous_echo, t):
    """me: where you are. radar_echo: where the NPC is, measured from the base station.
    previous_echo: the echo from one step ago (None at the very start). t: the time.
    Give back (go, facing): which way you move (length 1 or less) and which way you look (length 1)."""
    move = radar_echo - me                   # the move from you to the NPC
    facing = move / np.linalg.norm(move)     # look straight at it
    go = facing                              # and walk the way you are looking, at full speed
    return go, facing


chase(my_rule)

Now make it yours. Change the rule or the numbers, and run `chase(my_rule)` again.

1. Why did the first rule work, even though you are slower? Look at where the NPC goes.
2. Try smaller values of `my_speed`: $0.3$, $0.2$, $0.15$, $0.1$, $0.05$. Does slower always mean you miss? Look at the pictures to explain what happens.
3. **Aim ahead.** The NPC's velocity is `(radar_echo - previous_echo) / step_time`. Head for where it will be a moment from now, but keep **facing** where it is now. Does aiming ahead help or hurt? What happens at the corners?
4. **The lazy rule.** Walk to one spot on the track and wait there: `go` is $(0, 0)$, and you still face the NPC. Is that a good plan? Why does it work on this track?
5. Change `step_time` to $0.5$, then to $1$. Same rule, same speeds. What changes, and why?
6. Write a rule that runs **away** from the NPC, and make `my_speed` big. Watch the safety limit stop the chase.
7. Set `npc_shape = "box"` and chase again. Then make the box tall and thin, or short and wide, with `touch_box`. Which shape is easiest to catch, and why?

In [ ]:
# your rule here

## Deep 10: Did they meet?

In Deep 9, `closest_during_step` (for a circle) and `caught_in_box` (for a box) decide whether you caught the NPC. Here you build them yourself.

During one step you walk in a straight line from $\mathbf{a}_0$ to $\mathbf{a}_1$, and at the same time the NPC runs from $\mathbf{b}_0$ to $\mathbf{b}_1$. Call a moment inside the step $s$, from $0$ (the start of the step) to $1$ (the end). At moment $s$ you are at $\mathbf{a}_0 + s(\mathbf{a}_1 - \mathbf{a}_0)$, and the NPC is at $\mathbf{b}_0 + s(\mathbf{b}_1 - \mathbf{b}_0)$.

1. **Picture it first.** As the NPC runs during one step, its touch circle sweeps out a long rectangle with round ends. If your path never enters that shape, you surely missed. But entering it is not enough to catch the NPC. Why not? Draw two paths that cross the shape, one that catches the NPC and one that does not. The hard part is **the same moment**.
2. **Ride along with the NPC.** Pretend you are sitting on the NPC. From there the NPC never moves, and only you seem to move. Show that the move from you to the NPC at moment $s$ is $\mathbf{r}_0 + s\mathbf{d}$, where $\mathbf{r}_0 = \mathbf{b}_0 - \mathbf{a}_0$ and $\mathbf{d} = (\mathbf{b}_1 - \mathbf{b}_0) - (\mathbf{a}_1 - \mathbf{a}_0)$. (The move from you to the NPC is "NPC minus you".)
3. Being on exactly the same spot at the same moment means $\mathbf{r}_0 + s\mathbf{d} = (0, 0)$. That is two equations, one for each slot, but only one unknown, $s$. Why does that almost never have an answer? That is why the chase has a touch range.
4. **A box touch zone.** Say the NPC's touch zone is a box, $W$ to each side and $H$ up and down. You catch it when the first slot of the move is between $-W$ and $W$ **and** the second slot is between $-H$ and $H$. Write $\mathbf{r}_0 = (p, q)$ and $\mathbf{d} = (u, w)$. Then you catch it at moment $s$ when
   $$-W \le p + su \le W \qquad -H \le q + sw \le H \qquad 0 \le s \le 1$$
   That is a **system of linear inequalities** in $s$, and "at the same moment" is built in: it is the same $s$ in every inequality. Solve each one for $s$. Be careful: dividing by a negative number flips $\le$ around, and if $u$ is $0$ there is nothing to divide by. Each inequality cuts the possible values of $s$ down. You caught the NPC if some $s$ is left over.
5. Finish `my_caught_in_box` below, and test it on the five cases. When $W = H = R$ the box is a square, and its rule is "$L_\infty$ size at most $R$" (Part 0.3, "Beyond the coins"). Which case does the square catch, but the circle of radius $R$ miss? Why does the square catch whenever the circle does?
6. **The circle.** For the round touch zone the size squared is $\langle \mathbf{r}_0 + s\mathbf{d}, \mathbf{r}_0 + s\mathbf{d} \rangle = \langle \mathbf{r}_0, \mathbf{r}_0 \rangle + 2s\langle \mathbf{r}_0, \mathbf{d} \rangle + s^2 \langle \mathbf{d}, \mathbf{d} \rangle$. That is not linear in $s$: it has an $s^2$, and its graph is a curve with a lowest point. Find that lowest point in `closest_during_step`. Why does the code then keep $s$ between $0$ and $1$?
7. **Now in 3D.** Move the chase into a Roblox world, where a spot is $(x, y, z)$ and $y$ is up. The box becomes a box with depth, $W$, $H$ and $D$. How many inequalities do you need now? The circle becomes a **ball**. Does the formula for the closest moment change at all? Change `my_caught_in_box` so it works for 3D moves, and test it on a move that passes over the NPC's head.

In [ ]:
def my_caught_in_box(r0, d, W, H):
    """True if, at some moment s from 0 to 1, slot 0 of r0 + s * d is between -W and W
    and slot 1 is between -H and H."""
    low = 0.0                    # the moments s that are still possible: from low ...
    high = 1.0                   # ... to high
    for slot in range(2):
        p = r0[slot]
        u = d[slot]
        half = [W, H][slot]
        # -half <= p + s * u <= half: work out which moments s this allows,
        # and move low up or high down to match. Careful when u is 0 or negative!
        # finish me
    return low <= high           # is any moment left?


cases = [
    (np.array([1.0, 0.0]), np.array([-2.0, 0.0])),      # head on
    (np.array([1.0, 1.0]), np.array([0.0, 0.0])),       # nobody gets closer
    (np.array([0.5, 0.2]), np.array([-1.0, 0.0])),      # passes 0.2 away
    (np.array([-0.415, 0.585]), np.array([1.0, -1.0])), # passes close, on a slant
    (np.array([0.05, 0.0]), np.array([3.0, 0.0])),      # starts inside the zone
]
for r0, d in cases:
    square = my_caught_in_box(r0, d, 0.1, 0.1)
    circle = closest_during_step(r0, r0 + d) <= 0.1
    print("yours:", square, "  Deep 9's box:", caught_in_box(r0, r0 + d, 0.1, 0.1), "  circle:", circle)

<p><a href="https://colab.research.google.com/github/frcusaca/admath/blob/jia/notebooks/01_rover_moves_linear_combinations.ipynb" target="_blank" rel="noopener" title="Next: Notebook 1: The Rover&#x27;s Moves (linear combinations) (opens in a new tab)">Notebook 1 <b style="color:#a8327d">&gt;&gt;</b></a></p>

<small>Copyright © 2026-present CAehi.us. Licensed under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).</small>